# Phase 2: Statistics & Hypothesis Testing

**Overarching question:** What factors are associated with inspection failures and higher violation counts among Chicago food establishments?

In [1]:
import sqlite3
import pandas as pd

In [2]:
conn = sqlite3.connect('chicago_food_safety.db')

In [3]:
h1=pd.read_sql_query("""
SELECT Risk, Results
FROM Inspections
WHERE Results IN ('Pass', 'Fail', 'Pass w/ Conditions')
    AND "Risk" IN ('Risk 1 (High)', 'Risk 2 (Medium)', 'Risk 3 (Low)')
""",conn)
contingency_h1=pd.crosstab(h1['Risk'],h1['Results'])
contingency_h1

Results,Fail,Pass,Pass w/ Conditions
Risk,,,
Risk 1 (High),44764,124127,36306
Risk 2 (Medium),10602,28200,8303
Risk 3 (Low),5073,10370,2066


In [4]:
h2 = pd.read_sql_query("""
    SELECT i."Inspection ID", i."Facility Type",
           COUNT(v."Violations_code") AS violation_count
    FROM "Inspections" i
    LEFT JOIN "violations" v ON i."Inspection ID" = v."Inspection ID"
    WHERE i."Facility Type" IS NOT NULL
    GROUP BY i."Inspection ID", i."Facility Type"
    ORDER BY i."Inspection ID"
""", conn)
top_facility=h2["Facility Type"].value_counts().head(6).index
h2_top=h2[h2["Facility Type"].isin(top_facility)].copy()
h2_top

,Inspection ID,Facility Type,violation_count
0,44247,Restaurant,6
1,44248,Restaurant,3
2,44249,Restaurant,1
5,44252,Restaurant,3
6,44253,Restaurant,4
...,...,...,...
308997,2642298,Grocery Store,0
308998,2642300,School,4
308999,2642303,Daycare Above and Under 2 Years,5
309000,2642305,Grocery Store,0


In [5]:
h3=pd.read_sql_query(""" 
SELECT "Inspection Date",Results
FROM Inspections 
WHERE "Results" IN ('Pass', 'Fail', 'Pass w/ Conditions')""", conn, parse_dates=['Inspection Date'])
h3['season']=h3['Inspection Date'].dt.month.isin([12,1,2]).map({True:'Winter',False:"Other Seasons"})
h3['passed']=h3['Results'].isin(['Pass','Pass w/ Conditions'])
summary_h3=h3.groupby('season')['passed'].agg(['sum','count'])
summary_h3

,sum,count
season,,
Other Seasons,162144,209561
Winter,47228,60250


In [6]:
h4 = pd.read_sql_query("""
    SELECT i."Inspection ID", i."Inspection Date",
           COUNT(v."Violations_code") AS violation_count
    FROM "Inspections" i
    LEFT JOIN "violations" v ON i."Inspection ID" = v."Inspection ID"
    GROUP BY i."Inspection ID", i."Inspection Date"
""", conn, parse_dates=['Inspection Date'])

h4['year'] = h4['Inspection Date'].dt.year
h4[['year', 'violation_count']]

,year,violation_count
0,2010,6
1,2010,3
2,2010,1
3,2010,4
4,2010,4
...,...,...
314172,2026,0
314173,2026,4
314174,2026,5
314175,2026,0


In [11]:
conn.execute("""
    CREATE INDEX IF NOT EXISTS idx_inspections_license_date_result
    ON "Inspections" ("License #", "Inspection Date", "Results")
""")
conn.commit()
h5=pd.read_sql_query(""" 
SELECT "Inspection ID", "Results" as current, CASE WHEN EXISTS (
SELECT 1 FROM Inspections prev
WHERE prev."License #"=curr."License #"
AND prev."Inspection Date"<curr."Inspection Date"
AND prev."Results"='Fail') THEN 'prior_fail' ELSE 'no_prior_fail' END AS prior
FROM Inspections curr
WHERE curr."Results" IN ('Pass', 'Fail', 'Pass w/ Conditions')""",conn)
contingency_h5=pd.crosstab(h5['current'],h5['prior'])
contingency_h5

prior,no_prior_fail,prior_fail
current,,
Fail,25813,34626
Pass,62755,99942
Pass w/ Conditions,16548,30127
